<a href="https://colab.research.google.com/github/vhgauto/2026-Escuela-de-Primavera/blob/main/EdP2026_PRISMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on: PRISMA Processing

---
[Ver repositorio en GitHub](https://github.com/vhgauto/2026-Escuela-de-Primavera)

---

Ejecutar en la terminal el siguiente comando:

`sudo apt install gdal-bin libabsl-dev libnode-dev libproj-dev`

In [ ]:
# 25min
install.packages("tidyverse", dependencies = TRUE)
install.packages("tidyterra", dependencies = TRUE)
install.packages("RStoolbox", dependencies = TRUE)

Los paquetes más relevantes son [terra](https://rspatial.org/) para el procesamiento de datos espaciales, [tidyverse](https://tidyverse.tidyverse.org/) para generar figuras y procesamiento general, [tidyterra](https://dieghernan.github.io/tidyterra/) que permite confeccionar mapas, [prismaread](https://irea-cnr-mi.github.io/prismaread/) para transformar el producto PRISMA y [RStoolbox](https://bleutner.github.io/RStoolbox/) para análisis de sensado remoto.

In [ ]:
library(terra)
library(tidyverse)
library(tidyterra)

El producto PRISMA, nivel de procesamiento L2D en reflectancia de superficie, del 2022-03-11, se extrae y convierte de formato (`.he5` a `.tif`) para facilitar su lectura.

Se elige que la totalidad del producto se convierta a `.tif`, existiendo la posibilidad de extraer el rango VNIR, el SWIR o ambos unidos.

Además del ráster, se generan dos archivos de texto: `.ang` contiene datos de la geometría de adquisición y `.wvl` posee las longitudes de onda y la posición de cada banda espectral. Estos datos serán útiles para confeccionar firmas espectrales.

Incorporo el ráster generado y el vector del Embalse San Roque, para luego recortar. Verifico la cantidad de bandas disponibles en el recorte.

In [ ]:
r <- rast("https://github.com/vhgauto/2026-Escuela-de-Primavera/raw/refs/heads/main/tp_prisma/salida/20220311.tif")
nlyr(r)

Leo el vector que contiene los sitios de muestreo para análisis de endmembers.

In [ ]:
p <- vect("https://github.com/vhgauto/2026-Escuela-de-Primavera/raw/refs/heads/main/tp_prisma/vectores/puntos.geojson")

Son cuatro sitios correspondientes a coberturas de suelo urbano, agua, algas y vegetación.

Verifico creando un mapa RGB incluyendo los sitios. Las bandas del rojo, verde y azul corresponden con las capas 34, 20 y 10, respectivamente.

In [ ]:
r_fecha <- sources(r) |>
  basename() |>
  ymd()

ggplot() +
  geom_spatraster_rgb(
    data = r,
    r = 34,
    g = 20,
    b = 10,
    stretch = "lin"
  ) +
  geom_spatvector(data = p, shape = 21, fill = "red", color = "gold") +
  geom_spatvector_label(
    data = p,
    aes(label = puntos),
    size = 2,
    vjust = -.25,
    border.color = NA,
    fill = "white"
  ) +
  labs(title = r_fecha, x = NULL, y = NULL) +
  coord_sf(expand = FALSE) +
  theme_minimal(base_size = 8) +
  theme_sub_plot(background = element_blank()) +
  theme_sub_panel(background = element_blank())

# Firmas espectrales

Primeramente, se leen las longitudes de onda de cada banda junto con el número de banda a partir del archivo de texto (.wvl) generado.

In [ ]:
wvl <- read_delim(
  "https://github.com/vhgauto/2026-Escuela-de-Primavera/raw/refs/heads/main/tp_prisma/salida/PRS_L2D_STD_20220311142257_20220311142301_0001_HCO_FULL.wvl",
  delim = " ",
  show_col_types = FALSE
) |>
  rename(banda = band) |>
  select(wl, banda)

Luego, se combina con las reflectancias de superficies extraídas del producto PRISMA. Se acomodan los datos y se conserva únicamente la etiqueta del sitio, el número de banda, la reflectancia de superficie y la longitud de onda (`wl`) en nm.

In [ ]:
reflect_tbl <- terra::extract(r, p, bind = TRUE) |>
  as_tibble() |>
  pivot_longer(
    cols = starts_with("PRS_L2D_STD_"),
    names_to = "banda",
    values_to = "reflect"
  ) |>
  mutate(banda = str_extract(banda, "\\d+$")) |>
  mutate(banda = as.numeric(banda)) |>
  inner_join(wvl, by = join_by(banda)) |>
  select(puntos, banda, reflect, wl)

head(reflect_tbl)

Se grafica la firma espectral para identificar las diferencias entre los sitios seleccionados.



In [ ]:
ggplot(reflect_tbl, aes(wl, reflect, group = puntos, color = puntos)) +
  geom_line(linewidth = .5) +
  scale_x_continuous(breaks = scales::breaks_width(200)) +
  scale_color_brewer(palette = "Dark2") +
  labs(x = "Long. de onda (nm)", y = "Reflect.", color = NULL) +
  theme_bw() +
  theme_sub_axis(text = element_text(color = "black")) +
  theme_sub_panel(
    grid.minor = element_blank(),
    grid.major = element_line(linewidth = .1, color = "grey"),
    background = element_blank()
  ) +
  theme_sub_strip(text = element_text(color = "black", face = "bold")) +
  theme_sub_plot(background = element_blank(), margin = margin(r = 10)) +
  theme_sub_legend(
    position = "top",
    key.spacing.x = unit(20, "pt"),
    background = element_blank(),
    text = element_text(margin = margin())
  )

Las diferencias entre los cuatro sitios se hace evidente entre los rangos 400-1800 nm y entre 1950-2500 nm, donde las firmas espectrales se separan.

# Spectral Angle Mapper (SAM)

El análisis SAM el rango espectral entre los 400 y 1800 nm. Se calcula el ángulo de distancia en el espacio espectral para identificar los patrones de cobertura.

Se identifican los números de bandas involucradas en el rango propuesto.

In [ ]:
banda_rango <- wvl |>
  filter(between(wl, 400, 1800)) |>
  pull(banda) |>
  range()
banda_rango

Se seleccionan las bandas comprendidas entre las posiciones 1 y 143. Asimismo, conservo el mismo rango para las firmas espectrales.

In [ ]:
r_vnir <- r[[banda_rango[1]:banda_rango[2]]]
reflect_vnir <- terra::extract(r_vnir, p)
rownames(reflect_vnir) <- p$puntos

Cálculo del SAM y obtención de los valores de los ángulos de separación espectrales.



In [ ]:
lago_sam_ang <- RStoolbox::sam(r_vnir, reflect_vnir, angles = TRUE)

Verifico visualizando el raster obtenido. Cada panel muestra el ángulo de separación entre cada píxel del recorte y el endmember correspondiente. A menor ángulo (color rojo), más similitud.

In [ ]:
plot(
  lago_sam_ang,
  nc = 2,
  nr = 2,
  axes = FALSE,
  box = TRUE,
  col = rev(viridis::magma(500))
)

Es posible obtener la clasificación final de cada píxel del recorte, según los endmembers.



In [ ]:
lago_sam_clas <- RStoolbox::sam(r_vnir, reflect_vnir, angles = FALSE)

Inspección del resultado de la clasificación.



In [ ]:
ggplot() +
  geom_spatraster(data = lago_sam_clas) +
  scale_fill_princess_c(
    palette = "maori",
    breaks = 1:4,
    labels = p$puntos,
    direction = -1,
    guide = "legend"
  ) +
  labs(fill = NULL) +
  coord_sf(expand = FALSE) +
  labs(title = r_fecha) +
  theme_minimal(base_size = 8) +
  theme_sub_legend(
    position = "top",
    key.spacing.x = unit(15, "pt"),
    text = element_text(margin = margin(l = 3))
  ) +
  theme_sub_panel(
    background = element_rect(fill = "grey95"),
    grid.major = element_line(linetype = 2, color = "grey80", linewidth = .3)
  ) +
  theme_sub_plot(background = element_blank()) +
  theme_sub_axis_left(text = element_text(angle = 90, hjust = .5))

# Principal Component Analysis (PCA)

El análisis PCA se aplica sobre los píxeles de agua para identificar los tres principales.

Inicialmente, se calcula el MNDWI para conservar el agua en el recorte.

In [ ]:
mndwi <- (r[[34]] - r[[179]]) / (r[[34]] + r[[179]])
m <- thresh(mndwi)
m[isTRUE(m)] <- 1
m[isFALSE(m)] <- NA
m <- fillHoles(m)

Visualizo el recorte de agua para verificar.



In [ ]:
plot(m, col = "dodgerblue", axes = FALSE, box = TRUE, legend = FALSE)
north(xy = "topleft")

Obtengo el PCA del recorte, indicando que el ráster se compongan de los tres componentes más relevantes.

In [ ]:
agua_pca <- m * r_vnir
r_pca <- RStoolbox::rasterPCA(agua_pca, nComp = 3)

Visualizando por separado cada componente:



In [ ]:
plot(
  r_pca$map,
  col = viridis::magma(500),
  axes = FALSE,
  box = TRUE,
  nc = 3,
  nr = 1
)

La visualización RGB de los tres componentes principales genera la siguiente figura.



In [ ]:
plotRGB(stretch(r_pca$map), stretch = "lin", colNA = "transparent")
north(xy = "topleft")

Tomando el puntaje de los tres PCA más relevantes, la varianza acumulada alcanza el 75% del total:



In [ ]:
r_pca$model$sdev |>
  as_tibble() |>
  mutate(pca = paste0("PC", row_number())) |>
  mutate(pca = fct_inorder(pca)) |>
  mutate(aporte = value / sum(value)) |>
  slice_head(n = 3) |>
  mutate(acumulado = cumsum(aporte)) |>
  mutate(acumulado_label = paste0(round(acumulado * 100, 1), "%")) |>
  ggplot(aes(pca, aporte, fill = pca)) +
  geom_col() +
  geom_line(aes(y = acumulado, group = 1)) +
  geom_point(
    aes(y = acumulado, fill = pca),
    shape = 21,
    size = 3,
    stroke = 1,
    color = "white"
  ) +
  geom_text(aes(y = acumulado, label = acumulado_label), vjust = -1) +
  scale_y_continuous(
    labels = scales::label_percent(),
    breaks = scales::breaks_width(.1),
    # limits = c(0, .8)
  ) +
  scale_fill_brewer(palette = "Dark2", guide = guide_none()) +
  coord_cartesian(clip = "off") +
  labs(x = NULL, y = "Aporte") +
  theme_bw() +
  theme_sub_axis(text = element_text(color = "black")) +
  theme_sub_plot(background = element_blank()) +
  theme_sub_panel(background = element_blank())